# ContractRisk M3 — RAGAS 1.1 sobre las respuestas YA GUARDADAS de la corrida final

**Sesión nueva de Colab con GPU T4.** No regenera respuestas ni juicios del harness: lee `m3/results/final_m3/harness_rows.jsonl` (corrida original, que se conserva intacta) y recalcula RAGAS con la versión 1.1:

- segmentación de afirmaciones corregida (1.0 partía `CO1.PCCNTR.<n>` en sus puntos);
- variante **exacta del laboratorio S10** (context precision = fracción de pasajes relevantes para la pregunta; context recall = un sí/no sobre la referencia completa), y la **adaptación** de 1.0 (precisión ponderada por rango y recall por segmentos), reportadas por separado.

Juez: Qwen2.5-3B-Instruct local (mismo modelo del generador; dependencia declarada). Duración esperada: ~10 min. Devuelva el ZIP de resultados y el notebook con salidas, aunque falle.

In [1]:
import os, subprocess, sys, tempfile, zipfile
from pathlib import Path
# Opción A (recomendada si la rama ya está publicada): clonar el repositorio.
# Opción B: USE_ZIP = True y subir contractrisk_m3_ragas_v1_1_codigo.zip (generado con `git archive`).
USE_ZIP = True
REPO = 'https://github.com/vcastrop/Juan-Camilo-Ramon-Valentina-Castro.git'
BRANCH = 'Entrega/contractrisk-m3'
ROOT = Path('/content/contractrisk_m3_ragas_v1_1')
os.chdir('/content')  # el directorio actual puede no existir si la celda se re-ejecutó tras borrar ROOT
if ROOT.exists():
    raise RuntimeError('Use una sesión nueva: la carpeta ya existe y la evidencia anterior no se sobrescribe')
if USE_ZIP:
    from google.colab import files
    uploaded = files.upload()  # se guarda en /content
    if len(uploaded) != 1:
        raise ValueError('Suba únicamente contractrisk_m3_ragas_v1_1_codigo.zip')
    ROOT.mkdir(parents=True)
    with zipfile.ZipFile(Path('/content') / next(iter(uploaded))) as z:
        for member in z.infolist():
            if not (ROOT / member.filename).resolve().is_relative_to(ROOT.resolve()):
                raise ValueError('Ruta no permitida en ZIP')
        z.extractall(ROOT)
    COMMIT = z.comment.decode() or 'zip-sin-commit'
else:
    subprocess.run(['git', 'clone', '--depth', '1', '--branch', BRANCH, REPO, str(ROOT)], check=True)
    COMMIT = subprocess.run(['git', '-C', str(ROOT), 'rev-parse', 'HEAD'], capture_output=True, text=True, check=True).stdout.strip()
os.chdir(ROOT)
print('Código evaluado:', COMMIT)
assert Path('m3/rerun_ragas_m3.py').is_file(), 'El código no contiene la corrida final: use la rama o ZIP correctos'

Saving contractrisk_m3_ragas_v1_1_codigo.zip to contractrisk_m3_ragas_v1_1_codigo.zip
Código evaluado: 157590734c77a3f24754a847a9f465d2a98827cf


In [2]:
def run_visible(command, log_path):
    with open(log_path, 'w', encoding='utf-8') as log:
        proc = subprocess.Popen(command, stdout=subprocess.PIPE, stderr=subprocess.STDOUT, text=True, bufsize=1)
        for line in proc.stdout:
            print(line, end='', flush=True)
            log.write(line)
        code = proc.wait()
    if code:
        raise RuntimeError(f'Comando falló ({code}); consulte {log_path}')

run_visible([sys.executable, '-m', 'pip', 'install', '-r', 'm3/requirements-development-v2.txt'], 'installation.log')
run_visible([sys.executable, '-m', 'unittest', 'discover', '-s', 'm3/tests', '-v'], 'tests.log')
run_visible([sys.executable, '-c', "import sys; sys.path.insert(0,'m3'); from sqlite_support import ensure_sqlite; print(ensure_sqlite()); import torch; assert torch.cuda.is_available(), 'Active GPU T4'; print(torch.cuda.get_device_name(0))"], 'environment.log')

  Installing build dependencies: started
  Installing build dependencies: finished with status 'done'
  Getting requirements to build wheel: started
  Getting requirements to build wheel: finished with status 'done'
  Preparing metadata (pyproject.toml): started
  Preparing metadata (pyproject.toml): finished with status 'done'
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 12.0/12.0 MB 28.9 MB/s  0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 611.1/611.1 kB 27.2 MB/s  0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 1.4/1.4 MB 42.0 MB/s  0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 4.9/4.9 MB 75.3 MB/s  0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 566.4/566.4 kB 27.3 MB/s  0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 3.3/3.3 MB 77.8 MB/s  0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 7.5/7.5 MB 100.9 MB/s  0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 23.8/23.8 MB 138.3 MB/s  0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 1.0/1.0 MB 38.5 MB/s

In [3]:
run_visible([sys.executable, '-u', 'm3/rerun_ragas_m3.py', '--run', 'm3/results/final_m3', '--output', 'ragas_v1_1_resultados'], 'execution.log')

I0000 00:00:1793310404.862006    1292 cpu_feature_guard.cc:227] This TensorFlow binary is optimized to use available CPU instructions in performance-critical operations.
To enable the following instructions: AVX2 AVX512F FMA, in other operations, rebuild TensorFlow with the appropriate compiler flags.
I0000 00:00:1793310407.363831    1292 cudart_stub.cc:31] Could not find cuda drivers on your machine, GPU will not be used.
`torch_dtype` is deprecated! Use `dtype` instead!

Fetching 2 files: 100%|██████████| 2/2 [00:56<00:00, 28.03s/it]

Loading checkpoint shards: 100%|██████████| 2/2 [00:25<00:00, 12.61s/it]
RAGAS 1.1 rag_ingenuo m3_001
The following generation flags are not valid and may be ignored: ['temperature', 'top_p', 'top_k']. Set `TRANSFORMERS_VERBOSITY=info` for more details.
RAGAS 1.1 rag_ingenuo m3_002
RAGAS 1.1 rag_ingenuo m3_003
RAGAS 1.1 rag_ingenuo m3_004
RAGAS 1.1 rag_ingenuo m3_005
RAGAS 1.1 rag_ingenuo m3_006
RAGAS 1.1 rag_ingenuo m3_007
RAGAS 1.1 rag_ingenuo m3_008


In [4]:
from IPython.display import Markdown, display
display(Markdown(Path('ragas_v1_1_resultados/tablas_ragas_v1_1.md').read_text(encoding='utf-8')))

| Sistema | Faithfulness | Answer relevancy | Context precision (lab) | Context recall (lab) | Context precision (rango) | Context recall (segmentos) |
|---|---:|---:|---:|---:|---:|---:|
| RAG ingenuo (S07): denso top-3 | 0.697 (n=11) | 0.501 (n=10) | 0.000 (n=10) | 0.200 (n=10) | 0.033 (n=10) | 0.167 (n=10) |
| RAG avanzado: híbrido BM25+denso RRF con filtro de ID + reranking | 0.569 (n=12) | 0.317 (n=10) | 0.000 (n=10) | 0.500 (n=10) | 0.350 (n=10) | 0.592 (n=10) |
| Sistema final: avanzado + herramienta (agente acotado) | 0.653 (n=12) | 0.330 (n=10) | 0.000 (n=10) | 0.700 (n=10) | 0.383 (n=10) | 0.792 (n=10) |


In [5]:
# Ejecute también tras un fallo para conservar diagnóstico.
import shutil
from google.colab import files
result_dir = ROOT / 'ragas_v1_1_resultados'
result_dir.mkdir(exist_ok=True)
(result_dir / 'commit.txt').write_text(COMMIT + '\n', encoding='utf-8')
for log in ROOT.glob('*.log'):
    shutil.copy2(log, result_dir / log.name)
meta = result_dir / 'metadata.json'
print(meta.read_text(encoding='utf-8')[:3000] if meta.exists() else 'No se creó metadata: revisar logs.')
bundle = shutil.make_archive(str(ROOT / 'contractrisk_m3_ragas_v1_1_resultados'), 'zip', result_dir)
files.download(bundle)

{
  "status": "completed",
  "started_utc": "2026-10-29T21:46:32.879531+00:00",
  "ragas_version": "m3-ragas-lab-1.1",
  "applicability": {
    "faithfulness": "Salida válida y abstencion=false.",
    "answer_relevancy": "Salida válida en pregunta respondible.",
    "context_precision_lab": "Respondible con contexto.",
    "context_recall_lab": "Respondible con contexto.",
    "context_precision_rank": "Respondible con contexto.",
    "context_recall_segment": "Respondible con contexto."
  },
  "judge_model": [
    "Qwen/Qwen2.5-3B-Instruct",
    "aa8e72537993ba99e69dfaafa59ed015b17504d1"
  ],
  "embedding_model": [
    "sentence-transformers/paraphrase-multilingual-MiniLM-L12-v2",
    "e8f8c211226b894fcb81acc59f3b34ba3efd5f42"
  ],
  "source_run": "m3/results/final_m3",
  "source_sha256": "0966955b8df2968182b0813fcd645e76f00df6494523ca99ac5ed46767ca306e",
  "regenerated_answers": false,
  "gpu": "Tesla T4",
  "wall_seconds": 114.548572148,
  "finished_utc": "2026-10-29T21:50:19.750519

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>